In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvGP(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, M, padding=0):
        super().__init__()
        self.k = kernel_size
        self.P = in_channels * kernel_size * kernel_size
        self.M = M
        self.padding = padding
        # Parametri predefiniti del layer (Centri o Prototype)
        self.Z = nn.Parameter(torch.randn(M, self.P))
        self.U = nn.Parameter(torch.randn(M, out_channels))
        # Parametro di scala per RBF (gamma)
        self.gamma = nn.Parameter(torch.ones(1))

    def forward(self, x):
        # --- 1. GLOBAL MATRIXES ---
        d_global = torch.cdist(self.Z, self.Z)**2
        # (M, M)
        K_zz_inv = torch.linalg.inv(torch.exp(-self.gamma * d_global) + 1e-6 * torch.eye(self.M, device=x.device))

        # --- 2. UNFOLDING ---
        x_unf = F.unfold(x, kernel_size=self.k, padding=self.padding)
        L = x_unf.shape[-1] # num positions.
        # (B, L, P)
        patches = x_unf.transpose(1, 2) 

        # --- 3. LOCAL MATRIXES ---
        dist_sq = torch.cdist(patches, patches, p=2)**2
        # (B, L, L)
        K_ff = torch.exp(-self.gamma * dist_sq)

        dist_sq = torch.cdist(patches, self.Z.unsqueeze(0), p=2)**2
        # (B, L, M)
        K_fz = torch.exp(-self.gamma * dist_sq)

        # --- 4. GP PARAMETERS ---
        mu = K_fz @ K_zz_inv @ self.U
        
        sigma = K_ff - (K_fz @ K_zz_inv @ K_fz.transpose(1, 2))
        
        return mu, sigma

In [5]:
def test_conv_gp():
    # 1. Test parameters
    batch_size = 2
    in_channels = 3
    out_channels = 16
    h, w = 28, 28
    kernel_size = 3
    M = 20  # inducing points
    padding = 1
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 2. Input and model init.
    model = ConvGP(in_channels, out_channels, kernel_size, M, padding=padding).to(device)
    x = torch.randn(batch_size, in_channels, h, w).to(device)
    
    print(f"Input shape: {x.shape}")
    
    # 3. Forward Pass
    try:
        mu, sigma = model(x)
    except Exception as e:
        print(f"Error during forward pass: {e}")
        return

    # 4. Shapes
    expected_L = h * w 
    
    print("\n--- Verifying dims. ---")
    print(f"Mu shape: {mu.shape} | Expected: ({batch_size}, {expected_L}, {out_channels})")
    print(f"Sigma shape: {sigma.shape} | Expected: ({batch_size}, {expected_L}, {expected_L})")
    
    assert mu.shape == (batch_size, expected_L, out_channels), "Wrong Mu shape!"
    assert sigma.shape == (batch_size, expected_L, expected_L), "Wrong Sigma shape!"
    print("Shapes correct!")

    # 5. Numeric errors
    print("\n--- Verifying numeric errors ---")
    nan_mu = torch.isnan(mu).any()
    nan_sigma = torch.isnan(sigma).any()
    
    print(f"NaNs in Mu: {nan_mu}")
    print(f"NaNs in Sigma: {nan_sigma}")
    
    assert not nan_mu, "NaNs found in Mu!"
    assert not nan_sigma, "NaNs found in Sigma!"

    diag_sigma = torch.diagonal(sigma, dim1=-2, dim2=-1)
    min_var = diag_sigma.min().item()
    print(f"Minimum variance on the diagonal: {min_var:.8f}")
    
    assert min_var > -1e-5, "Negative variance found on the diagonal!"
    print("Numerically stable!")

    # 6. Backprop
    print("\n--- Verifying gradient ---")
    loss = mu.sum() + sigma.sum()
    loss.backward()
    
    grad_Z = model.Z.grad is not None
    grad_gamma = model.gamma.grad is not None
    print(f"Computed gradient for Z: {grad_Z}")
    print(f"Computed gradient for gamma: {grad_gamma}")
    
    assert grad_Z and grad_gamma, "Gradient not flowing!"
    print("Backpropagation working!")

if __name__ == "__main__":
    test_conv_gp()

Input shape: torch.Size([2, 3, 28, 28])

--- Verifying dims. ---
Mu shape: torch.Size([2, 784, 16]) | Expected: (2, 784, 16)
Sigma shape: torch.Size([2, 784, 784]) | Expected: (2, 784, 784)
Shapes correct!

--- Verifying numeric errors ---
NaNs in Mu: False
NaNs in Sigma: False
Minimum variance on the diagonal: 0.99997711
Numerically stable!

--- Verifying gradient ---
Computed gradient for Z: True
Computed gradient for gamma: True
Backpropagation working!
